In [ ]:
"""
Project entrypoint 
"""

from models import Patient, MedicalRecord
import pandas as pd
from data_manager import load_patient_data, get_critical_patients, save_patient_data, validate_name, validate_dob, update_patient, delete_patient
from hospital_analytics import plot_hospital_capacity
from billing import generate_invoice
from datetime import datetime


def display_menu():
    print("\n--- Hospital Database ---")
    print("1. View All Patients")
    print("2. View Critical Patients")
    print("3. Add Patient")
    print("4. Update Patient")
    print("5. Delete Patient")
    print("6. Generate Analytics Report")
    print("7. Generate Patient Invoice")
    print("8. Exit")

def main():
    print("=== Project Landing Page ===")
    print("Starting application...\n")

    filepath = 'patients.csv'
    patients_list = load_patient_data(filepath)
    db_running = True

    display_menu()

    while db_running:
        choice = input("\nSelect an option (1-8) or 'm' for menu: ")
        
        if choice == '1':
            print("\n--- Patient List ---")
            for idx, p in enumerate(patients_list, start=1):
                print(f"{idx}. {p}")
        elif choice == '2':
            print("\n--- Critical Patients ---")
            criticals = get_critical_patients(patients_list)
            if not criticals:
                print("No critical patients found.")
            else:
                for p in criticals:
                    print(f"CRITICAL: {p}")
        elif choice == '3':
            print("\n--- Add New Patient ---")
            name = input("Enter Name: ")
            if validate_name(name):
                dob = input("Enter DOB (YYYY-MM-DD): ")
                if validate_dob(dob):
                    diagnosis = input("Enter Diagnosis: ")
                    if not diagnosis.strip():
                        diagnosis = "Undiagnosed"
                    
                    is_critical = False
                    print("\nSelect Patient Status:")
                    print("1. Critical")
                    print("2. Non-Critical")
                    status_choice = input("Enter selection (1 or 2): ")
                    if status_choice == '1':
                        is_critical = True
                    
                    record = MedicalRecord(diagnosis, is_critical, 0.0)
                    new_p = Patient(len(patients_list) + 1, name, dob, record)
                    patients_list.append(new_p)
                    save_patient_data(filepath, patients_list)
                    print("Patient added successfully!")
        elif choice == '6':
            print("\nGenerating analytics report...")
            plot_hospital_capacity(filepath)
        elif choice == '7':
            print("\n--- Generate Patient Invoice ---")
            for idx, p in enumerate(patients_list, start=1):
                print(f"{idx}. {p.name}")
            try:
                p_idx = int(input("Select patient number: ")) - 1
                if 0 <= p_idx < len(patients_list):
                    patient = patients_list[p_idx]
                    invoice_text, amt_due = generate_invoice(patient)
                    print("\n" + invoice_text)
                else:
                    print("Invalid patient selection.")
            except ValueError:
                print("Please enter a valid number.")
        elif choice == '4':
            print("\n--- Update Patient ---")
            try:
                pid = int(input("Enter Patient ID to update: "))
                name = input("Enter New Name: ")
                if validate_name(name):
                    dob = input("Enter New DOB (YYYY-MM-DD): ")
                    if validate_dob(dob):
                        diagnosis = input("Enter New Diagnosis: ")
                        if not diagnosis.strip():
                            diagnosis = "Undiagnosed"
                        
                        is_critical = False
                        print("\nSelect New Patient Status:")
                        print("1. Critical")
                        print("2. Non-Critical")
                        status_choice = input("Enter selection (1 or 2): ")
                        if status_choice == '1':
                            is_critical = True
                        
                        if update_patient(patients_list, pid, name, dob, diagnosis, is_critical):
                            save_patient_data(filepath, patients_list)
                            print(f"Patient {pid} updated successfully!")
                        else:
                            print(f"Patient ID {pid} not found.")
            except ValueError:
                print("Invalid input. Patient ID must be a number.")
        elif choice == '5':
            print("\n--- Delete Patient ---")
            try:
                pid = int(input("Enter Patient ID to delete: "))
                if delete_patient(patients_list, pid):
                    save_patient_data(filepath, patients_list)
                    print(f"Patient {pid} deleted successfully!")
                else:
                    print(f"Patient ID {pid} not found.")
            except ValueError:
                print("Invalid input. Patient ID must be a number.")
        elif choice == '8':
            print("Shutting down system...")
            db_running = False
        elif choice.lower() == 'm':
            display_menu()
        else:
            print(f"\n[ERROR] '{choice}' is an invalid selection. Please choose 1-8.")
            display_menu()
    

if __name__ == "__main__":
    main()